# 通用整数结果检查模板（离线）
默认不导入 PYNQ，不操作硬件；演示数据均为 Python 合成。

In [ ]:
from pathlib import Path
import sys
repo = Path.cwd()
if repo.name == "refsim":
    repo = repo.parent
assert (repo / "refsim" / "compare_integer_results.py").is_file(), "请将 repo 设置为仓库根目录"
sys.path.insert(0, str(repo / "refsim"))
from compare_integer_results import compare
from run_generic_compare_demo import cases


## 两种算法的合成演示
accepted=True 表示符合测试预期；故意改错的样例应返回 mismatch。

In [ ]:
for name, inputs, expected, actual, wanted in cases():
    result = compare(expected, actual)
    print(name, result["status"], "accepted=", result["status"] == wanted)
    assert result["status"] == wanted


## 接收已采集结果的入口
此函数只比较，不采集硬件数据。独立计算 expected，由经验证的适配层生成 actual。source 是使用者声明，不是硬件证据。

In [ ]:
def check_collected_results(expected, actual, source):
    if not isinstance(source, str) or not source.strip():
        raise ValueError("必须说明实际结果来源")
    report = compare(expected, actual)
    report["actual_source_declared"] = source
    report["hardware_provenance_verified"] = False
    return report

example = check_collected_results([11,22,33,44], [11,22,33,45], "synthetic_manual_error")
assert example["mismatched"] == 1
print(example)


## 真板接入前（本轮未执行）
确认兼容镜像、PYNQ 版本、自有 Overlay；准备匹配的 bit/hwh 与接口契约。适配层需处理位宽、符号、顺序、有限等待、资源释放及必要的缓存一致性。保存独立参考、原始硬件输出与转换规则。
当前项目是纯 PL UART，不能直接假设 AXI DMA。详见 generic_compare_README.md；离线执行不证明 PYNQ 或跨板可用。